In [ ]:
from pyspark.errors import PySparkException
from py4j.protocol import Py4JJavaError
from pyspark.sql.functions import current_timestamp
from pyspark.sql import functions as F

In [ ]:
# Usuário, senha, host e port do banco de dados
usuario = dbutils.secrets.get("databases_lh_nautical", "sql_server_user")
senha = dbutils.secrets.get("databases_lh_nautical", "sql_server_password")
host = dbutils.secrets.get("databases_lh_nautical", "sql_server_host")
port = dbutils.secrets.get("databases_lh_nautical", "sql_server_port")

In [ ]:
def consulta_tabela_jdbc(jdbc_url, propriedades_conexao, nome_tabela_completa, numero_particao, coluna_referencia_paralelizacao = None):

    if coluna_referencia_paralelizacao is not None:
        limites_df = spark.read.jdbc(
            url = jdbc_url,
            table = f"(SELECT MIN({coluna_referencia_paralelizacao}) AS min_atributo, MAX({coluna_referencia_paralelizacao}) AS max_atributo FROM {nome_tabela_completa}) AS bounds",
            properties = propriedades_conexao
        )

        limites = limites_df.collect()[0]
        limite_inferior = limites['min_atributo']
        limite_superior = limites['max_atributo']

        df_temp = spark.read.jdbc(
            url = jdbc_url, 
            table = nome_tabela_completa,
            column = coluna_referencia_paralelizacao,
            lowerBound = limite_inferior if limite_inferior is not None else None,
            upperBound = limite_superior if limite_superior is not None else None,
            numPartitions = numero_particao, 
            properties = propriedades_conexao
        )
    else:
        df_temp = spark.read.jdbc(
            url = jdbc_url, 
            table = nome_tabela_completa,
            properties = propriedades_conexao
        )
    
    return df_temp

In [ ]:
def ingestao_incremental(jdbc_url, propriedades_conexao, nome_tabela, nome_catalog, coluna_referencia_paralelizacao = None):
    
    nome_tabela_origem = f"dbo.{nome_tabela}"
    nome_tabela_destino = f"{nome_catalog}.{nome_tabela}"

    # Verificar se a tabela de destino já existe no catálogo
    if not spark.catalog.tableExists(nome_tabela_destino):
        # Se a tabela não existe, então faz a carga da tabela
        df_temp = consulta_tabela_jdbc(jdbc_url, propriedades_conexao, nome_tabela_origem, coluna_referencia_paralelizacao)
        df_temp_com_data = df_temp.withColumn("data_ingestao", current_timestamp())\
            .withColumn("acao", F.lit("CARGA_INICIAL"))
                
        print(f"Primeira ingestão da tabela {nome_tabela} integrada com sucesso!")
            
    else:
        print(f"A tabela {nome_tabela} já existe")
        print("Vamos verificar se há dados novos ou se houve alguma update")
        df_temp = consulta_tabela_jdbc(jdbc_url, propriedades_conexao, nome_tabela_origem, coluna_referencia_paralelizacao)
        # Se a tabela existir
        # Pegar o número da última versão salva do delta lake
        ultima_versao = spark.sql(f"DESCRIBE HISTORY {nome_tabela_destino}").first()["version"]
        df_destino_ultimo = spark.read.format("delta") \
            .option("versionAsOf", ultima_versao) \
            .table(nome_tabela_destino) \
            .drop('data_ingestao', "acao")
                
        # Comparar a origem com a última foto da tabela
        df_novos_e_alterados = df_temp.exceptAll(df_destino_ultimo)
        # contar os numero de alterações
        numero_alteracoes = df_novos_e_alterados.count()
        
        if numero_alteracoes > 0:
            df_temp_com_data = df_novos_e_alterados \
                .withColumn("data_ingestao", current_timestamp()) \
                .withColumn("acao", "INSERT_OU_UPDATE")
            print(f"Foram adicionados {numero_alteracoes} na {nome_tabela}")
        
        else:
            df_temp_com_data = df_temp.exceptAll(df_temp)
            print("Nenhuma alteração detectada na origem.")

    return df_temp_com_data

In [ ]:
# Info do banco de dados
jdbc_url = f"jdbc:sqlserver://{host}:{port};databaseName=lh_nautical;encrypt=false;trustServerCertificate=true;"
propriedades_conexao = {
    "user": usuario,
    "password": senha,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

# Info do catalog do desenvolvedor
nome_catalog = dbutils.widgets.get("catalog_name")
schema = dbutils.widgets.get("schema")
numero_particao = dbutils.widgets.get("numPartitions")

In [ ]:
# Query para descobrir todas as tabelas do esquema 'dbo'
query_tabelas = "(select table_name from information_schema.tables where table_schema = 'dbo' and table_type = 'Base table') as tabelas"

try:
  # Tentaiva de conexão com o banco para obter as lista de tabelas para um DataFrame e convertemos para uma lista em Python
  df_tabelas = consulta_tabela_jdbc(jdbc_url, propriedades_conexao, query_tabelas, numero_particao)

  tabelas = df_tabelas.collect() 
  lista_tabelas = [row['table_name'] for row in df_tabelas.collect()]

except Py4JJavaError as e:
  erro_msg = str(e)
  print("[Erro de conexão/JDBC] - Falha ao conectar ou executar a query no banco de dados." )
  print(f"Detalhes do erro: {erro_msg}")
  lista_tabelas = []
  raise erro_msg
  
except PySparkException as e:
  erro_msg = str(e)
  print(f"[Erro do Spark] - Ocorreu uma falha interna no Spark: {erro_msg}")
  lista_tabelas = []
  raise erro_msg

except Exception as e:
  erro_msg = str(e)
  print(f"[Erro Inesperado] - {erro_msg}")
  lista_tabelas = []
  raise erro_msg


In [ ]:
# Ingestão de dados
tabelas_com_erro = []

for nome_tabela in lista_tabelas:
    print(f"Importanto a tabela: {nome_tabela}...")
    nome_catalog_completo = f"{nome_catalog}.{schema}"

    try:
        if nome_tabela not in ['variant_attribute_values', 'product_suppliers', 'stock_levels']:
            df_aux = ingestao_incremental(jdbc_url, propriedades_conexao, nome_tabela, nome_catalog_completo, "id")

        elif nome_tabela in ['product_suppliers', 'stock_levels']:
            df_aux = ingestao_incremental(jdbc_url, propriedades_conexao, nome_tabela, nome_catalog_completo, "updated_at")

        else:
            df_aux = ingestao_incremental(jdbc_url, propriedades_conexao, nome_tabela, nome_catalog_completo)

        # Adiciona a coluna com a data e hora atual do Spark
        df_aux.write.format("delta").mode("append").saveAsTable(f"{nome_catalog_completo}.{nome_tabela}")

    except Py4JJavaError as e:
        erro_msg = str(e)
        print("[Erro de conexão/JDBC] - Falha ao conectar ou executar a query no banco de dados." )
        print(f"Erro ao processar tabela {nome_tabela}: {erro_msg}")
        # Registra a tabela que falhou para o resumo final
        tabelas_com_erro.append(nome_tabela)
        
        # Dispara o e-mail de alerta
        raise erro_msg
    
    except PySparkException as e:
        erro_msg = str(e)
        print(f"[Erro do Spark] - Ocorreu uma falha interna no Spark: {erro_msg}")
        print(f"Erro ao processar tabela {nome_tabela}: {erro_msg}")

        # Registra a tabela que falhou para o resumo final
        tabelas_com_erro.append(nome_tabela)

        # Dispara o e-mail de alerta
        raise erro_msg
    
    except Exception as e:
        erro_msg = str(e)
        print(f"[Erro Inesperado] - {erro_msg}")

        # Registra a tabela que falhou para o resumo final
        tabelas_com_erro.append(nome_tabela)

        # Dispara o e-mail de alerta
        raise erro_msg


print("Ingestão finalizada!")
if tabelas_com_erro:
    print(f"Atenção! As seguintes tabelas falharam e geraram alertas: {tabelas_com_erro}")
else:
    print("Sucesso absoluto! Todas as tabelas foram copiadas sem erros.")